In [1]:
import sys
import os
import time
import threading

%load_ext autoreload
%autoreload 2

project_root = os.path.abspath('./fraud_detecton_project')
if project_root not in sys.path:
    sys.path.append(project_root)

print("Project root appended to sys.path. Ready for imports.")

Project root appended to sys.path. Ready for imports.


## Real Time Ingestor

In [2]:
from IngestionLayer.real_time_ingestor import RealTimeIngestor

KAFKA_BROKER = 'localhost:9092'
basic_csv_path = "data/5000_New_Basic.csv" 
advanced_csv_path = "data/sample_1500_sorted.csv"

print("--- Starting Kafka Producers ---")
streamer_a = RealTimeIngestor(bootstrap_servers=KAFKA_BROKER, topic='financial-transactions')
streamer_b = RealTimeIngestor(bootstrap_servers=KAFKA_BROKER, topic='ip-account-detection')

thread_a = threading.Thread(target=streamer_a.stream, args=(basic_csv_path, 0.2))
thread_b = threading.Thread(target=streamer_b.stream, args=(advanced_csv_path, 0.2))

thread_a.start()
thread_b.start()
print("Kafka Producers are now streaming data in the background.")

2026-04-03 14:08:15,260 - INFO - <BrokerConnection client_id=wsl-financial-transactions-admin, node_id=bootstrap-0 host=localhost:9092 <connecting> [IPv4 ('127.0.0.1', 9092)]>: connecting to localhost:9092 [('127.0.0.1', 9092) IPv4]
2026-04-03 14:08:15,262 - INFO - <BrokerConnection client_id=wsl-financial-transactions-admin, node_id=bootstrap-0 host=localhost:9092 <checking_api_versions_recv> [IPv4 ('127.0.0.1', 9092)]>: Broker version identified as 2.6
2026-04-03 14:08:15,263 - INFO - <BrokerConnection client_id=wsl-financial-transactions-admin, node_id=bootstrap-0 host=localhost:9092 <connected> [IPv4 ('127.0.0.1', 9092)]>: Connection complete.
2026-04-03 14:08:15,266 - INFO - <BrokerConnection client_id=wsl-financial-transactions-admin, node_id=0 host=Legion.localdomain:9092 <connecting> [IPv4 ('127.0.1.1', 9092)]>: connecting to Legion.localdomain:9092 [('127.0.1.1', 9092) IPv4]
2026-04-03 14:08:15,267 - INFO - <BrokerConnection client_id=wsl-financial-transactions-admin, node_id=

--- Starting Kafka Producers ---
Kafka Producers are now streaming data in the background.


2026-04-03 14:08:15,382 - INFO - <BrokerConnection client_id=wsl-financial-transactions-producer, node_id=0 host=Legion.localdomain:9092 <connecting> [IPv4 ('127.0.1.1', 9092)]>: connecting to Legion.localdomain:9092 [('127.0.1.1', 9092) IPv4]
2026-04-03 14:08:15,383 - INFO - <BrokerConnection client_id=wsl-financial-transactions-producer, node_id=0 host=Legion.localdomain:9092 <connected> [IPv4 ('127.0.1.1', 9092)]>: Connection complete.
2026-04-03 14:08:15,384 - INFO - <BrokerConnection client_id=wsl-financial-transactions-producer, node_id=bootstrap-0 host=localhost:9092 <connected> [IPv4 ('127.0.0.1', 9092)]>: Closing connection. 
2026-04-03 14:08:15,403 - INFO - <BrokerConnection client_id=wsl-ip-account-detection-producer, node_id=0 host=Legion.localdomain:9092 <connecting> [IPv4 ('127.0.1.1', 9092)]>: connecting to Legion.localdomain:9092 [('127.0.1.1', 9092) IPv4]
2026-04-03 14:08:15,404 - INFO - <BrokerConnection client_id=wsl-ip-account-detection-producer, node_id=0 host=Legi

## Real Time Processing

In [ ]:
from RealTimeProcessing.RealTime import RealTimeFraudDetection
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

brokers = "localhost:9092"
general_topic = "financial-transactions"
input_topic = "ip-account-detection"
    
susp_receivers_topic = "suspicious-receivers"
susp_receivers_hdfs_path = "hdfs://localhost:9000/user/student/assignment/realtime_v37/data/suspicious_receivers"
susp_receivers_checkpoint = "hdfs://localhost:9000/user/student/assignment/realtime_v37/checkpoint_dir/kafka/suspicious_receivers"
    
acc_takeover_topic = "ip-takeover"
acc_takeover_hdfs_path = "hdfs://localhost:9000/user/student/assignment/realtime_v37/data/ip_takeover"
acc_takeover_checkpoint = "hdfs://localhost:9000/user/student/assignment/realtime_v37/checkpoint_dir/kafka/ip_takeover"

potential_offenders_topic = "potential-offenders"
potential_offenders_hdfs_path = "hdfs://localhost:9000/user/student/assignment/realtime_v37/data/potential_offenders"
potential_offenders_checkpoint = "hdfs://localhost:9000/user/student/assignment/realtime_v37/checkpoint_dir/kafka/potential_offenders"

spark = (SparkSession.builder
         .appName("RealTimeFraudDetection")
         .master("local[*]")
         .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:3.5.1")
         .config("spark.streaming.stopGracefullyOnShutdown", True)
         .getOrCreate())

spark.sparkContext.setLogLevel("WARN")

pipeline = RealTimeFraudDetection(spark)
raw_df = pipeline.read_from_kafka(brokers, input_topic)
general_df = pipeline.read_from_kafka(brokers, general_topic)

susp_receivers_df = pipeline.detect_suspicious_receivers(raw_df)
susp_receivers_query = pipeline.write_aggregated_stream(susp_receivers_df, susp_receivers_topic, susp_receivers_hdfs_path, susp_receivers_checkpoint, brokers)

acc_takeover_df = pipeline.analyse_ip_takeover(raw_df)
acc_takeover_query = pipeline.write_aggregated_stream(acc_takeover_df, acc_takeover_topic, acc_takeover_hdfs_path, acc_takeover_checkpoint, brokers)

potential_offenders_df = pipeline.analyse_potential_offenders(general_df)
potential_offenders_query = pipeline.write_aggregated_stream(potential_offenders_df, potential_offenders_topic, potential_offenders_hdfs_path, potential_offenders_checkpoint, brokers)

print("Starting the streaming queries ...")
pipeline.await_termination()

26/04/03 14:08:18 WARN Utils: Your hostname, Legion resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/03 14:08:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/hduser/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/student/.ivy2/cache
The jars for the packages stored in: /home/student/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1ea8a964-3211-448f-8abc-40c272f4b75b;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;3.5.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;3.5.1 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.3 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#scala-parallel-collections_2.13;1.0.4 in central
	found org.apache.commons#commons-pool2

Initialising Spark Session ...
Reading from Kafka topic: ip-account-detection
Reading from Kafka topic: financial-transactions
Detecting receivers that received transactions from multiple senders within a short time frame...


2026-04-03 14:08:23,575 - INFO - Callback Server Starting
2026-04-03 14:08:23,576 - INFO - Socket listening on ('127.0.0.1', 34785)


Analysing different senders with the same IP address...
Performing analytics on the sender and receiver accounts...
Starting the streaming queries ...


2026-04-03 14:08:25,739 - INFO - Python Server ready to receive messages
2026-04-03 14:08:25,740 - INFO - Received command c on object id p2
2026-04-03 14:08:25,967 - INFO - Python Server ready to receive messages
2026-04-03 14:08:25,968 - INFO - Received command c on object id p0
2026-04-03 14:08:25,968 - INFO - Python Server ready to receive messages
2026-04-03 14:08:25,970 - INFO - Received command c on object id p1
2026-04-03 14:08:45,104 - INFO - Received command c on object id p1             
2026-04-03 14:08:45,111 - INFO - Received command c on object id p0
2026-04-03 14:08:45,447 - INFO - Received command c on object id p2
2026-04-03 14:09:10,059 - INFO - Received command c on object id p2             
2026-04-03 14:09:10,115 - INFO - Received command c on object id p1
2026-04-03 14:09:10,124 - INFO - Received command c on object id p0
2026-04-03 14:09:35,047 - INFO - Received command c on object id p2             
2026-04-03 14:09:35,114 - INFO - Received command c on object 

## Batch Ingestor

In [6]:
from IngestionLayer.batch_ingestor import BatchIngestor

LOCAL_CSV_FILE = "data/sample_100000_sorted.csv" 
HDFS_RAW_ZONE = "assignment/raw/financial_transactions"

print("--- Starting Batch Ingestion ---")
batch_ingestor = BatchIngestor(hdfs_base_dir=HDFS_RAW_ZONE)
batch_ingestor.ingest_file(local_file_path=LOCAL_CSV_FILE)

2026-04-03 13:52:10,865 - INFO - Ensuring HDFS directory exists: assignment/raw/financial_transactions


--- Starting Batch Ingestion ---


2026-04-03 13:52:12,274 - INFO - HDFS directory is ready.
2026-04-03 13:52:12,275 - INFO - Starting batch ingestion: data/sample_100000_sorted.csv -> assignment/raw/financial_transactions/sample_100000_sorted.csv
2026-04-03 13:52:13,887 - INFO - Successfully ingested sample_100000_sorted.csv to HDFS.
2026-04-03 13:52:15,238 - INFO - Verification passed: assignment/raw/financial_transactions/sample_100000_sorted.csv exists in HDFS. Size: 15809022 bytes.


## Batch Processing

In [7]:
import os
import sys

print("--- Starting Batch Processing & Results Viewer ---")

project_root = os.getcwd()

try:
    batch_dir = os.path.join(project_root, 'BatchProcessing')
    os.chdir(batch_dir)
    
    if batch_dir not in sys.path:
        sys.path.insert(0, batch_dir)

    from main_batch import BatchPipeline
    print("Executing Batch Pipeline...")
    batch_pipeline = BatchPipeline()
    batch_pipeline.execute_pipeline()
    print("Batch Pipeline Finished.\n")
    
    from show_result import BatchResultsViewer
    print("Fetching and Displaying Results...")
    viewer = BatchResultsViewer()
    viewer.display_aggregations()
    
    print("\nBatch Processing & Viewing Completed Successfully.")

except Exception as e:
    print(f"An error occurred: {e}")

finally:
    os.chdir(project_root)
    print(f"\nReturned to root directory: {os.getcwd()}")

--- Starting Batch Processing & Results Viewer ---
Executing Batch Pipeline...


26/04/03 13:52:17 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.



BEFORE: RAW DATAFRAME SCHEMA & DATA
root
 |-- transaction_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- sender_account: string (nullable = true)
 |-- receiver_account: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- location: string (nullable = true)
 |-- device_used: string (nullable = true)
 |-- is_fraud: boolean (nullable = true)
 |-- fraud_type: string (nullable = true)
 |-- time_since_last_transaction: double (nullable = true)
 |-- spending_deviation_score: double (nullable = true)
 |-- velocity_score: integer (nullable = true)
 |-- geo_anomaly_score: double (nullable = true)
 |-- payment_channel: string (nullable = true)
 |-- ip_address: string (nullable = true)
 |-- device_hash: string (nullable = true)

TOTAL ROWS BEFORE CLEANING: 100000
+--------------+--------------------+--------------+----------------+------+----------------+-

26/04/03 13:52:20 WARN MemoryManager: Total allocation exceeds 50.00% (477,364,224 bytes) of heap memory
Scaling row group sizes to 88.92% for 4 writers


Batch Pipeline Finished.

Fetching and Displaying Results...

1. TRANSACTION VOLUME AGGREGATION
+----------------+------------------+-------------+
|transaction_type|total_transactions| total_volume|
+----------------+------------------+-------------+
|         deposit|             25017|2.496204238E7|
|      withdrawal|             24850|    7598665.4|
|         payment|             25216|   2547424.43|
|        transfer|             24917|    631842.73|
+----------------+------------------+-------------+


2. FRAUD PROFILES AGGREGATION
+----------------+-----------+-----------+----------------+
|transaction_type|fraud_count|total_count|fraud_percentage|
+----------------+-----------+-----------+----------------+
|         deposit|        923|      25017|          3.6895|
|         payment|        925|      25216|          3.6683|
|        transfer|        898|      24917|           3.604|
|      withdrawal|        878|      24850|          3.5332|
+----------------+-----------+------

In [14]:
import os; print(os.getcwd())

/home/student/de-prj/fraud_detection_project


## MongoDB

In [16]:
from MongoDB.run_serving import ServingLayerRunner
from MongoDB.mongodb_connector import MongoDBConnector

print("--- Starting MongoDB Serving Layer ---")

mongo_runner = ServingLayerRunner('BatchProcessing/config.json')

try:
    mongo_runner.execute_ingestion()
    mongo_runner.execute_queries_and_display()
finally:
    mongo_runner.shutdown()

print("MongoDB Serving Layer Completed.")

--- Starting MongoDB Serving Layer ---
Ingesting curated data into MongoDB...


26/04/03 14:05:30 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Running complex analytical aggregations...

--- Query 1: Cross-Referenced Real-Time Threats ---
+-------+--------------------+---------------+-------+----------------+-------------+--------------+-----------------------+
|amount |composite_risk_score|ip_address     |ip_flag|receiver_account|receiver_flag|sender_account|threat_level           |
+-------+--------------------+---------------+-------+----------------+-------------+--------------+-----------------------+
|142.4  |89.0                |2.130.26.252   |false  |ACC451752       |true         |ACC969823     |CRITICAL (MULTI-VECTOR)|
|2502.69|41.8                |185.20.14.77   |false  |ACC999992       |false        |ACC300001     |HIGH (ANOMALY)         |
|2199.33|40.6                |185.20.14.77   |false  |ACC999992       |false        |ACC300002     |HIGH (ANOMALY)         |
|12.2   |39.8                |19.3.62.20     |false  |ACC481166       |false        |ACC527141     |HIGH (ANOMALY)         |
|10.83  |39.8                

## Neo4J

In [4]:
from Neo4J.neo4j_loader import Neo4jLoader, read_hdfs
from auradb import NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD

print("--- Starting Neo4j Serving Layer ---")
neo4j_loader = Neo4jLoader(NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD)

try:
    print("Clearing Neo4j and Creating Indexes...")
    neo4j_loader.clear_database()
    neo4j_loader.create_indexes()

    print("Loading transactions into Graph...")
    tx_data = read_hdfs("/user/student/assignment/realtime_v36/data/potential_offenders")
    if tx_data:
        neo4j_loader.insert_transactions(tx_data)
        neo4j_loader.insert_potential_fraud(tx_data)

    print("Loading suspicious receivers into Graph...")
    susp_data = read_hdfs("/user/student/assignment/realtime_v36/data/suspicious_receivers")
    if susp_data:
        neo4j_loader.insert_suspicious_receivers(susp_data)

    print("Loading IP takeover into Graph...")
    ip_data = read_hdfs("/user/student/assignment/realtime_v36/data/ip_takeover")
    if ip_data:
        for r in ip_data: 
            if "accounts" not in r or not isinstance(r["accounts"], list):
                r["accounts"] = []
        neo4j_loader.insert_ip_takeover(ip_data)

    print("All data SUCCESSFULLY loaded into Neo4j")
finally:
    neo4j_loader.close()

--- Starting Neo4j Serving Layer ---
Clearing Neo4j and Creating Indexes...
Loading transactions into Graph...
Loading suspicious receivers into Graph...
Loading IP takeover into Graph...
All data SUCCESSFULLY loaded into Neo4j


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import Row
from auradb import *
from neo4j import GraphDatabase

URI = NEO4J_URI
USERNAME = NEO4J_USERNAME
PASSWORD = NEO4J_PASSWORD

spark = SparkSession.builder \
    .appName("FraudDetectionQueries") \
    .getOrCreate()

driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD),
    connection_timeout=30,
    max_connection_lifetime=1000,
    keep_alive=True
)
import json
from pyspark.sql.types import *

def query_potential_fraud(tx):
    query = """
    MATCH (ip:IP)<-[:USES_IP]-(a:SenderAccount)
    RETURN ip.address AS IP, 
           count(a) AS Num_Accounts,
           collect(a.id) AS Account_List
    ORDER BY Num_Accounts DESC
    """
    result = tx.run(query)
    return [record.data() for record in result]

with driver.session() as session:
    data = session.execute_read(query_potential_fraud)
    
schema = StructType([
    StructField("IP", StringType(), True),
    StructField("Num_Accounts", IntegerType(), True),
    StructField("Account_List", StringType(), True)
])

for r in data:
    r["Account_List"] = json.dumps(r["Account_List"])

print("Query 1: IP Takeover Accounts")
df1 = spark.createDataFrame(data, schema=schema)
df1.show(truncate=False)

def query_fraud_senders(tx):
    query = """
    MATCH (s:SenderAccount)-[t:TRANSFER]->(r:ReceiverAccount:SuspiciousReceiver)
    RETURN
        r.id AS Suspicious_Receiver,
        count(t) AS Receive_Count,
        sum(toFloat(t.amount)) AS Total_Amount_Received,
        collect(DISTINCT s.id) AS  Sender_List
    ORDER BY Total_Amount_Received DESC
    """
    result = tx.run(query)
    return [record.data() for record in result]

with driver.session() as session:
    data = session.execute_read(query_fraud_senders)

schema = StructType([
    StructField("Suspicious_Receiver", StringType(), True),
    StructField("Receive_Count", IntegerType(), True),
    StructField("Total_Amount_Received", DoubleType(), True),
    StructField("Sender_List", ArrayType(StringType()), True),
])
print("Query 2: Suspicious Receiver with Extreme High Amount")
df = spark.createDataFrame(data, schema=schema)
df.show(truncate=False)

In [15]:
from pyspark.sql import SparkSession
from pyspark.sql import Row
from Neo4J.auradb import *
from neo4j import GraphDatabase
import json
from pyspark.sql.types import *
from Neo4J.neo4j_query import Neo4jQuery

URI = NEO4J_URI
USERNAME = NEO4J_USERNAME
PASSWORD = NEO4J_PASSWORD

spark = SparkSession.builder \
    .appName("FraudDetectionQueries") \
    .getOrCreate()

driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD),
    connection_timeout=30,
    max_connection_lifetime=1000,
    keep_alive=True
)

with driver.session() as session:
    data = session.execute_read(Neo4jQuery.query_potential_fraud)

for r in data:
    r["Account_List"] = json.dumps(r["Account_List"])

print("Query 1: IP Takeover Accounts")
df1 = spark.createDataFrame(data, schema=Neo4jQuery.first_schema)
df1.show(truncate=False)

with driver.session() as session:
    data = session.execute_read(Neo4jQuery.query_fraud_senders)

print("Query 2: Suspicious Receiver with Extreme High Amount")
df = spark.createDataFrame(data, schema=Neo4jQuery.second_schema)
df.show(truncate=False)

Query 1: IP Takeover Accounts
+---------------+------------+----------------------------------------------------+
|IP             |Num_Accounts|Account_List                                        |
+---------------+------------+----------------------------------------------------+
|61.210.47.148  |4           |["ACC778625", "ACC277084", "ACC919296", "ACC415254"]|
|247.107.135.49 |4           |["ACC523903", "ACC247306", "ACC890183", "ACC205599"]|
|126.68.187.146 |3           |["ACC626368", "ACC853008", "ACC983123"]             |
|117.189.225.182|3           |["ACC797062", "ACC669015", "ACC652877"]             |
|54.146.165.3   |3           |["ACC384399", "ACC790029", "ACC866104"]             |
|210.7.106.179  |3           |["ACC861474", "ACC187067", "ACC440125"]             |
|180.33.34.125  |2           |["ACC877453", "ACC563155"]                          |
|31.177.80.146  |2           |["ACC249077", "ACC568828"]                          |
|126.162.188.145|2           |["ACC771343", "A